# PlantVillage — Apple & Tomato+Pepper Classifiers (ConvNeXtBase)
**Structure:** PlantVillage/plantvillage dataset/color/Class___Name/images  
**Model 1:** Apple — 4 classes  
**Model 2:** Tomato + Bell Pepper — all classes combined into one classifier  
**Backbone:** ConvNeXtBase (local weights)  
**Protocol:** 5-fold stratified CV, CLAHE preprocessing, 3-phase training  
**Cell 10:** Evaluation on saved models using same 5-fold splits

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import os
import cv2
import gc
import glob
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support
)
import seaborn as sns
import matplotlib.pyplot as plt

print(f'TensorFlow: {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── Cell 2: CONFIG ────────────────────────────────────────────────────────────
CONFIG = {
    # Single PlantVillage color folder — contains ALL crop subfolders
    'color_dir':             r'C:\path\to\PlantVillage\plantvillage dataset\color',

    # CLAHE output directories (created automatically)
    'apple_clahe_dir':       r'C:\path\to\clahe\apple',
    'tomato_pepper_clahe_dir': r'C:\path\to\clahe\tomato_pepper',

    # ConvNeXtBase local weights (.h5)
    'convnext_weights':      r'C:\path\to\convnext_base_notop.h5',

    # Where to save models, JSON, confusion matrices
    'save_dir':              r'C:\path\to\save\models',

    # Training
    'img_size':              384,
    'batch_size':            32,
    'num_folds':             5,
    'seed':                  42,

    # Phase 1 — frozen backbone
    'epochs_phase1':         25,
    'lr_phase1':             1e-4,
    'patience_phase1':       7,

    # Phase 2A — partial unfreeze
    'epochs_phase2a':        20,
    'lr_phase2a':            1e-5,
    'patience_phase2a':      7,

    # Phase 2B — full unfreeze
    'epochs_phase2b':        10,
    'lr_phase2b':            5e-6,
    'patience_phase2b':      6,
}

tf.random.set_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
os.makedirs(CONFIG['save_dir'],                exist_ok=True)
os.makedirs(CONFIG['apple_clahe_dir'],         exist_ok=True)
os.makedirs(CONFIG['tomato_pepper_clahe_dir'], exist_ok=True)
print('CONFIG loaded')

In [ ]:
# ── Cell 3: Mixed precision ───────────────────────────────────────────────────
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
print('Mixed precision: mixed_float16 enabled')

In [ ]:
# ── Cell 4: Build dataframes ──────────────────────────────────────────────────
def build_df_from_prefixes(color_dir, prefixes):
    """
    Reads the PlantVillage color folder and filters classes by one or more prefixes.
    prefixes: list of strings e.g. ['Tomato', 'Pepper']
    Labels are assigned 0,1,2... across all matched classes sorted alphabetically.
    Returns: df with [image_path, image_id, label, class_name], class_names list
    """
    all_dirs = sorted([
        d for d in os.listdir(color_dir)
        if os.path.isdir(os.path.join(color_dir, d))
        and any(d.startswith(p) for p in prefixes)
    ])

    if not all_dirs:
        raise ValueError(f'No folders found matching prefixes {prefixes} in {color_dir}')

    print(f'Classes found ({prefixes}):')
    label_map = {name: idx for idx, name in enumerate(all_dirs)}
    rows = []

    for class_name in all_dirs:
        class_dir = os.path.join(color_dir, class_name)
        images = (
            glob.glob(os.path.join(class_dir, '*.jpg'))  +
            glob.glob(os.path.join(class_dir, '*.jpeg')) +
            glob.glob(os.path.join(class_dir, '*.JPG'))  +
            glob.glob(os.path.join(class_dir, '*.png'))
        )
        print(f'  {label_map[class_name]}: {class_name} — {len(images)} images')
        for img_path in images:
            rows.append({
                'image_path': img_path,
                'image_id':   os.path.basename(img_path),
                'label':      label_map[class_name],
                'class_name': class_name
            })

    df = pd.DataFrame(rows).reset_index(drop=True)
    return df, all_dirs

# Apple — 4 classes
df_apple, apple_classes = build_df_from_prefixes(
    CONFIG['color_dir'], ['Apple'])
print(f'\nApple total: {len(df_apple)} images, {len(apple_classes)} classes')

print()

# Tomato + Bell Pepper — combined into one classifier
df_tomato_pepper, tomato_pepper_classes = build_df_from_prefixes(
    CONFIG['color_dir'], ['Tomato', 'Pepper'])
print(f'\nTomato+Pepper total: {len(df_tomato_pepper)} images, {len(tomato_pepper_classes)} classes')

In [ ]:
# ── Cell 5: CLAHE preprocessing ──────────────────────────────────────────────
def apply_clahe(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img[:, :, 0] = clahe.apply(img[:, :, 0])
    img = cv2.cvtColor(img, cv2.COLOR_LAB2RGB)
    return img

def run_clahe(df, clahe_dir, name):
    print(f'Applying CLAHE to {name}...')
    skipped     = 0
    clahe_paths = []
    for idx, row in df.iterrows():
        out_path = os.path.join(clahe_dir, row['image_id'])
        if not os.path.exists(out_path):
            img = apply_clahe(row['image_path'])
            if img is not None:
                cv2.imwrite(out_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            else:
                skipped += 1
        clahe_paths.append(out_path)
        if idx % 500 == 0:
            print(f'  {idx}/{len(df)}')
    print(f'{name} CLAHE done — skipped {skipped} corrupt')
    return clahe_paths

df_apple['clahe_path'] = run_clahe(
    df_apple, CONFIG['apple_clahe_dir'], 'Apple')
df_tomato_pepper['clahe_path'] = run_clahe(
    df_tomato_pepper, CONFIG['tomato_pepper_clahe_dir'], 'Tomato+Pepper')

# Filter missing/corrupt
df_apple         = df_apple[df_apple['clahe_path'].apply(os.path.exists)].reset_index(drop=True)
df_tomato_pepper = df_tomato_pepper[df_tomato_pepper['clahe_path'].apply(os.path.exists)].reset_index(drop=True)
print(f'Valid Apple:         {len(df_apple)}')
print(f'Valid Tomato+Pepper: {len(df_tomato_pepper)}')

In [ ]:
# ── Cell 6: Preprocessing, augmentation, dataset, model functions ─────────────
def preprocess_convnext(image_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [CONFIG['img_size'], CONFIG['img_size']])
    img.set_shape([CONFIG['img_size'], CONFIG['img_size'], 3])
    img = tf.keras.applications.convnext.preprocess_input(img)
    return img, label

def augment_image_tf(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.2)
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
    img = tf.image.random_saturation(img, lower=0.8, upper=1.2)
    crop_size = tf.cast(
        tf.cast(CONFIG['img_size'], tf.float32) *
        tf.random.uniform([], 0.85, 1.0), tf.int32)
    img = tf.image.random_crop(img, size=[crop_size, crop_size, 3])
    img = tf.image.resize(img, [CONFIG['img_size'], CONFIG['img_size']])
    return img, label

def build_dataset(image_paths, labels, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    ds = ds.map(preprocess_convnext, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_image_tf, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(1024)
    ds = ds.batch(CONFIG['batch_size']).prefetch(tf.data.AUTOTUNE)
    return ds

def build_dataset_eval(image_paths, labels):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    ds = ds.map(preprocess_convnext, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(CONFIG['batch_size']).prefetch(tf.data.AUTOTUNE)
    return ds

def build_model(num_classes, trainable_base=False):
    base = tf.keras.applications.ConvNeXtBase(
        input_shape=(CONFIG['img_size'], CONFIG['img_size'], 3),
        include_top=False,
        weights=None
    )
    base.load_weights(
        CONFIG['convnext_weights'],
        by_name=True,
        skip_mismatch=True
    )
    base.trainable = trainable_base
    inputs  = tf.keras.Input(shape=(CONFIG['img_size'], CONFIG['img_size'], 3))
    x       = base(inputs, training=False)
    x       = tf.keras.layers.GlobalAveragePooling2D()(x)
    x       = tf.keras.layers.LayerNormalization()(x)
    x       = tf.keras.layers.Dropout(0.3)(x)
    x       = tf.keras.layers.Dense(128, activation='relu')(x)
    x       = tf.keras.layers.Dropout(0.2)(x)
    x       = tf.keras.layers.Dense(num_classes)(x)
    outputs = tf.keras.layers.Activation('softmax', dtype='float32')(x)
    return tf.keras.Model(inputs, outputs)

print('All functions ready')

In [ ]:
# ── Cell 7: Training function ─────────────────────────────────────────────────
def train_crop(df, class_names, crop_name):
    print(f'\n{"#"*60}')
    print(f'# Training: {crop_name}  |  {len(class_names)} classes')
    print(f'{"#"*60}')

    num_classes = len(class_names)

    cw = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(df['label']),
        y=df['label']
    )
    class_weight_dict = dict(enumerate(cw))
    print(f'Class weights: {class_weight_dict}')

    skf = StratifiedKFold(
        n_splits=CONFIG['num_folds'],
        shuffle=True,
        random_state=CONFIG['seed']
    )

    fold_accuracies = []
    fold_results    = []
    all_preds       = []
    all_labels      = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(df['clahe_path'], df['label'])
    ):
        print(f'\n{"="*55}')
        print(f'{crop_name} — FOLD {fold+1}/{CONFIG["num_folds"]}')
        print(f'{"="*55}')

        train_df    = df.iloc[train_idx]
        val_df      = df.iloc[val_idx]
        train_ds    = build_dataset(train_df['clahe_path'].values,
                                    train_df['label'].values, augment=True)
        val_ds      = build_dataset(val_df['clahe_path'].values,
                                    val_df['label'].values, augment=False)
        val_ds_eval = build_dataset_eval(val_df['clahe_path'].values,
                                         val_df['label'].values)

        # Phase 1
        print('\n[Phase 1] Frozen backbone')
        model = build_model(num_classes, trainable_base=False)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(CONFIG['lr_phase1']),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        model.fit(
            train_ds, validation_data=val_ds,
            epochs=CONFIG['epochs_phase1'],
            class_weight=class_weight_dict,
            callbacks=[
                tf.keras.callbacks.EarlyStopping(
                    monitor='val_accuracy', mode='max',
                    patience=CONFIG['patience_phase1'],
                    restore_best_weights=True),
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor='val_loss', factor=0.2,
                    patience=4, min_lr=1e-7, verbose=1)
            ], verbose=1
        )

        # Phase 2A
        print('\n[Phase 2A] Partial unfreeze — top 30 layers')
        base_model = model.layers[1]
        for layer in base_model.layers[-30:]:
            layer.trainable = True
        model.compile(
            optimizer=tf.keras.optimizers.Adam(
                CONFIG['lr_phase2a'], clipnorm=1.0),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        model.fit(
            train_ds, validation_data=val_ds,
            epochs=CONFIG['epochs_phase2a'],
            class_weight=class_weight_dict,
            callbacks=[
                tf.keras.callbacks.EarlyStopping(
                    monitor='val_accuracy', mode='max',
                    patience=CONFIG['patience_phase2a'],
                    restore_best_weights=True),
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor='val_loss', factor=0.2,
                    patience=4, min_lr=1e-7, verbose=1)
            ], verbose=1
        )

        # Phase 2B
        print('\n[Phase 2B] Full unfreeze')
        for layer in base_model.layers:
            layer.trainable = True
        model.compile(
            optimizer=tf.keras.optimizers.Adam(
                CONFIG['lr_phase2b'], clipnorm=1.0),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        model.fit(
            train_ds, validation_data=val_ds,
            epochs=CONFIG['epochs_phase2b'],
            class_weight=class_weight_dict,
            callbacks=[
                tf.keras.callbacks.EarlyStopping(
                    monitor='val_accuracy', mode='max',
                    patience=CONFIG['patience_phase2b'],
                    restore_best_weights=True),
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor='val_loss', factor=0.2,
                    patience=3, min_lr=1e-7, verbose=1)
            ], verbose=1
        )

        # Save
        save_path = os.path.join(
            CONFIG['save_dir'],
            f'fold_{fold+1}_{crop_name.lower().replace("+","_")}_convnext.keras'
        )
        model.save(save_path)
        print(f'Saved: {save_path}')

        # Evaluate
        model.compile(
            optimizer=tf.keras.optimizers.Adam(1e-5),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        probs  = model.predict(val_ds_eval, verbose=0)
        preds  = np.argmax(probs, axis=1)
        labels = val_df['label'].values
        acc    = np.mean(preds == labels)

        p, r, f1, _ = precision_recall_fscore_support(
            labels, preds,
            labels=list(range(num_classes)),
            average=None
        )

        fold_accuracies.append(float(acc))
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

        fold_results.append({
            'fold':     fold + 1,
            'accuracy': float(acc),
            'per_class': {
                class_names[i]: {
                    'precision': float(p[i]),
                    'recall':    float(r[i]),
                    'f1':        float(f1[i])
                } for i in range(num_classes)
            },
            'macro': {
                'precision': float(np.mean(p)),
                'recall':    float(np.mean(r)),
                'f1':        float(np.mean(f1))
            }
        })

        print(f'\nFold {fold+1} Accuracy: {acc:.4f}')
        print(classification_report(labels, preds, target_names=class_names))

        del model
        tf.keras.backend.clear_session()
        gc.collect()

    # Final summary
    print(f'\n{"="*55}')
    print(f'FINAL — {crop_name}')
    print(f'{"="*55}')
    print(f'Per-fold: {[f"{a:.4f}" for a in fold_accuracies]}')
    print(f'Mean: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})')
    print(f'\nAll folds combined:')
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # Save JSON
    output = {
        'crop':          crop_name,
        'backbone':      'convnext',
        'img_size':      CONFIG['img_size'],
        'num_classes':   num_classes,
        'class_names':   class_names,
        'num_folds':     CONFIG['num_folds'],
        'mean_accuracy': float(np.mean(fold_accuracies)),
        'std_accuracy':  float(np.std(fold_accuracies)),
        'folds':         fold_results
    }
    json_path = os.path.join(
        CONFIG['save_dir'],
        f'{crop_name.lower().replace("+","_")}_results_convnext.json'
    )
    with open(json_path, 'w') as f:
        json.dump(output, f, indent=2)
    print(f'JSON saved: {json_path}')

    # Confusion matrix
    cm     = confusion_matrix(all_labels, all_preds)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    fig_w  = max(14, len(class_names) * 1.2)
    fig, axes = plt.subplots(1, 2, figsize=(fig_w * 2, fig_w * 0.7))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[0])
    axes[0].set_title('Counts')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].tick_params(axis='x', rotation=45)

    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[1])
    for t in axes[1].texts:
        t.set_text(t.get_text() + '%')
    axes[1].set_title('% per class')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    axes[1].tick_params(axis='x', rotation=45)

    plt.suptitle(
        f'{crop_name} — ConvNeXtBase\n'
        f'Mean Acc: {np.mean(fold_accuracies):.4f} '
        f'(±{np.std(fold_accuracies):.4f})',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    cm_path = os.path.join(
        CONFIG['save_dir'],
        f'{crop_name.lower().replace("+","_")}_confusion_convnext.png'
    )
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Confusion matrix saved: {cm_path}')

    return output

print('Training function ready')

In [ ]:
# ── Cell 8: Train Apple ───────────────────────────────────────────────────────
apple_results = train_crop(df_apple, apple_classes, 'Apple')

In [ ]:
# ── Cell 9: Train Tomato + Bell Pepper ───────────────────────────────────────
tomato_pepper_results = train_crop(df_tomato_pepper, tomato_pepper_classes, 'Tomato+Pepper')

In [ ]:
# ── Cell 10: Evaluate saved models on same 5-fold splits ─────────────────────
# Run this cell independently to re-evaluate without retraining
# Set EVAL_CROP and EVAL_CLASSES to whichever model you want to evaluate

# ── CONFIG for evaluation — update as needed ──────────────────────────────────
EVAL_CROP    = 'Apple'            # 'Apple' or 'Tomato+Pepper'
EVAL_DF      = df_apple           # df_apple or df_tomato_pepper
EVAL_CLASSES = apple_classes      # apple_classes or tomato_pepper_classes
# Model filename pattern — must match what was saved in train_crop
# Apple:         fold_X_apple_convnext.keras
# Tomato+Pepper: fold_X_tomato_pepper_convnext.keras
MODEL_PREFIX = EVAL_CROP.lower().replace('+', '_')

print(f'Evaluating: {EVAL_CROP}')
print(f'Classes:    {EVAL_CLASSES}')
print(f'Model file: fold_X_{MODEL_PREFIX}_convnext.keras')

skf_eval = StratifiedKFold(
    n_splits=CONFIG['num_folds'],
    shuffle=True,
    random_state=CONFIG['seed']
)

eval_fold_accuracies = []
eval_fold_results    = []
eval_all_preds       = []
eval_all_labels      = []
num_classes          = len(EVAL_CLASSES)

for fold, (train_idx, val_idx) in enumerate(
    skf_eval.split(EVAL_DF['clahe_path'], EVAL_DF['label'])
):
    print(f'\n--- Evaluating Fold {fold+1}/5 ---')

    val_df = EVAL_DF.iloc[val_idx]

    model_path = os.path.join(
        CONFIG['save_dir'],
        f'fold_{fold+1}_{MODEL_PREFIX}_convnext.keras'
    )

    if not os.path.exists(model_path):
        print(f'  Model not found: {model_path} — skipping')
        continue

    model = tf.keras.models.load_model(model_path)
    print(f'  Loaded: {model_path}')

    ds_eval = build_dataset_eval(
        val_df['clahe_path'].values,
        val_df['label'].values
    )

    probs  = model.predict(ds_eval, verbose=1)
    preds  = np.argmax(probs, axis=1)
    labels = val_df['label'].values
    acc    = np.mean(preds == labels)

    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds,
        labels=list(range(num_classes)),
        average=None
    )

    eval_fold_accuracies.append(float(acc))
    eval_all_preds.extend(preds.tolist())
    eval_all_labels.extend(labels.tolist())

    eval_fold_results.append({
        'fold':     fold + 1,
        'accuracy': float(acc),
        'per_class': {
            EVAL_CLASSES[i]: {
                'precision': float(p[i]),
                'recall':    float(r[i]),
                'f1':        float(f1[i])
            } for i in range(num_classes)
        },
        'macro': {
            'precision': float(np.mean(p)),
            'recall':    float(np.mean(r)),
            'f1':        float(np.mean(f1))
        }
    })

    print(f'  Fold {fold+1} Accuracy: {acc:.4f}')
    print(classification_report(labels, preds, target_names=EVAL_CLASSES))

    del model
    tf.keras.backend.clear_session()
    gc.collect()

# Summary
print(f'\n{"="*55}')
print(f'EVALUATION SUMMARY — {EVAL_CROP}')
print(f'{"="*55}')
print(f'Per-fold: {[f"{a:.4f}" for a in eval_fold_accuracies]}')
print(f'Mean: {np.mean(eval_fold_accuracies):.4f} (+/- {np.std(eval_fold_accuracies):.4f})')
print(f'\nAll folds combined:')
print(classification_report(eval_all_labels, eval_all_preds, target_names=EVAL_CLASSES))

# Save JSON
eval_output = {
    'crop':          EVAL_CROP,
    'backbone':      'convnext',
    'img_size':      CONFIG['img_size'],
    'num_classes':   num_classes,
    'class_names':   EVAL_CLASSES,
    'num_folds':     len(eval_fold_accuracies),
    'mean_accuracy': float(np.mean(eval_fold_accuracies)),
    'std_accuracy':  float(np.std(eval_fold_accuracies)),
    'folds':         eval_fold_results
}
eval_json_path = os.path.join(
    CONFIG['save_dir'],
    f'{MODEL_PREFIX}_eval_results.json'
)
with open(eval_json_path, 'w') as f:
    json.dump(eval_output, f, indent=2)
print(f'\nEval JSON saved: {eval_json_path}')

# Confusion matrix
cm     = confusion_matrix(eval_all_labels, eval_all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
fig_w  = max(14, num_classes * 1.2)
fig, axes = plt.subplots(1, 2, figsize=(fig_w * 2, fig_w * 0.7))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EVAL_CLASSES, yticklabels=EVAL_CLASSES,
            ax=axes[0])
axes[0].set_title('Counts')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].tick_params(axis='x', rotation=45)

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=EVAL_CLASSES, yticklabels=EVAL_CLASSES,
            ax=axes[1])
for t in axes[1].texts:
    t.set_text(t.get_text() + '%')
axes[1].set_title('% per class')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle(
    f'{EVAL_CROP} — Evaluation on Saved Models\n'
    f'Mean Acc: {np.mean(eval_fold_accuracies):.4f} '
    f'(±{np.std(eval_fold_accuracies):.4f})',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
eval_cm_path = os.path.join(
    CONFIG['save_dir'],
    f'{MODEL_PREFIX}_eval_confusion.png'
)
plt.savefig(eval_cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Confusion matrix saved: {eval_cm_path}')